### Limpeza dos dados

In [1]:
import re
import spacy
from gensim.models import Word2Vec

nlp = spacy.load("pt_core_news_sm")


ficheiros = [
    "Harry Potter e A Pedra Filosofal.txt", 
    "Harry_Potter_Camara_Secreta-br.txt"
]

sentences = []

for nome_ficheiro in ficheiros:
    print(f"A processar: {nome_ficheiro}...")
    
    with open(nome_ficheiro, "r", encoding="utf8") as f:
        texto = f.read()
        
        texto = re.sub(r'\f', '', texto)
        
        doc = nlp(texto)
        
        for sent in doc.sents:
            tokens = [token.text.lower() for token in sent if not token.is_punct and not token.is_space]
            if len(tokens) > 0:
                sentences.append(tokens)

print(f"Total de frases carregadas: {len(sentences)}")

A processar: Harry Potter e A Pedra Filosofal.txt...
A processar: Harry_Potter_Camara_Secreta-br.txt...
Total de frases carregadas: 14610


# Modelo 1

In [2]:
model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=0, epochs=5, workers=3)

In [3]:
word_vectors = model.wv
word_vectors.save("word2vec.wordvectors")

#### Visualizar o modelo

In [4]:
model.wv.save_word2vec_format('model_harry.txt',binary=False)
#projector.tensorflow.org

In [5]:
!python -m gensim.scripts.word2vec2tensor -i model_harry.txt -o model_harry

2026-04-29 22:36:06,766 - word2vec2tensor - INFO - running C:\Users\helen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gensim\scripts\word2vec2tensor.py -i model_harry.txt -o model_harry
2026-04-29 22:36:06,766 - keyedvectors - INFO - loading projection weights from model_harry.txt
2026-04-29 22:36:07,713 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (6989, 100) matrix of type float32 from model_harry.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-04-29T22:36:07.653551', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'load_word2vec_format'}
2026-04-29 22:36:08,234 - word2vec2tensor - INFO - 2D tensor file saved to model_harry_tensor.tsv
2026-04-29 22:36:08,234 - word2vec2tensor - INFO - Tensor metadata file saved to model_harry_metadata.tsv
2026-04-29 22:36:08,234 

In [6]:
vetor = model.wv['harry']
print(vetor)

[-0.24705803  1.024848    0.4865377   0.22152846  0.46995232 -0.33644044
  0.43564242  1.2161319  -0.6715477  -0.36336148 -0.04330849 -0.9082329
 -0.35535344  0.46417528 -0.21787763 -0.19299611  0.6719911  -0.19523834
 -0.1068093  -1.1113319  -0.04390107 -0.30495888  0.69854623 -0.22227435
 -0.360473   -0.10494974  0.07513593  0.24182932 -0.26990235  0.15338095
  0.51212764 -0.46201593  0.44719332 -0.37228462 -0.17380314  0.94757015
  0.36635384 -0.5648083  -0.4748096  -0.63426304  0.22191793 -0.29364082
  0.15901369  0.5928318   0.7160546  -0.20750055 -1.1149796  -0.35545003
  0.44472316  0.7812651   0.3952291  -0.81750476  0.02125157 -0.15890726
 -0.647116    0.31356075 -0.09518822 -0.25749704 -0.50871015  0.03631191
  0.07154525 -0.25743636  0.5802448  -0.21942666 -0.87169784  0.21775582
  0.37032786  0.47791436 -0.8130093   0.5729847  -0.2073596   0.59660816
  0.38463008  0.1075596   0.37617314  0.49871096  0.36207664  0.18907513
 -0.09088937 -0.11644524 -0.4146985  -0.06211368 -0.

In [7]:
'varinha' in model.wv

True

#### Similariedade

In [8]:
model.wv.most_similar('harry')

[('hermione', 0.997395396232605),
 ('rony', 0.9944241046905518),
 ('voz', 0.9940207600593567),
 ('olhou', 0.9933691620826721),
 ('ansioso', 0.9924331903457642),
 ('apontando', 0.9923849701881409),
 ('quando', 0.9922526478767395),
 ('abriu', 0.9919499158859253),
 ('virou', 0.9913915395736694),
 ('fred', 0.9910944700241089)]

In [9]:
model.wv.most_similar('voldemort')

[('aquela', 0.9994578957557678),
 ('aquele', 0.9994328022003174),
 ('nos', 0.9993847012519836),
 ('algum', 0.9993670582771301),
 ('jamais', 0.9993225336074829),
 ('desde', 0.9993011355400085),
 ('esperando', 0.9992884993553162),
 ('minha', 0.9992820024490356),
 ('feito', 0.9992611408233643),
 ('esse', 0.9992605447769165)]

In [10]:
print(model.wv.similarity('harry', 'snape'))
print(model.wv.similarity('harry', 'hagrid'))
print(model.wv.similarity('rony', 'hermione'))
print(model.wv.similarity('harry', 'draco'))

0.9783963
0.98962104
0.99458593
0.9894008


In [11]:
pairs = [
    ('harry', 'rony'),          
    ('harry', 'hermione'),     
    ('harry', 'voldemort'),    
    ('dumbledore', 'snape'),   
    ('varinha', 'vassoura'),   
    ('grifinória', 'sonserina')
]

for w1, w2 in pairs:
    print('%s \t %s \t% .2f' % (w1, w2, model.wv.similarity(w1,w2)))

harry 	 rony 	 0.99
harry 	 hermione 	 1.00
harry 	 voldemort 	 0.97
dumbledore 	 snape 	 1.00
varinha 	 vassoura 	 1.00
grifinória 	 sonserina 	 1.00


#### Intruso

In [12]:
model.wv.doesnt_match(['harry', 'rony', 'hermione', 'draco']) 

'draco'

In [13]:
model.wv.doesnt_match(['grifinória', 'sonserina', 'corvinal', 'lufa-lufa', 'harry'])

'harry'

In [14]:
resultado = model.wv.most_similar(positive=['sonserina', 'harry'], negative=['grifinória'])
print("Harry - Grifinória + Sonserina =")
print(resultado)

Harry - Grifinória + Sonserina =
[('hagrid', 0.9930852055549622), ('mione', 0.9919682741165161), ('hermione', 0.9917145371437073), ('draco', 0.9898769855499268), ('ansioso', 0.9897860884666443), ('depressa', 0.9894287586212158), ('viu', 0.9890276789665222), ('percy', 0.98902428150177), ('malfoy', 0.9889043569564819), ('baixinho', 0.9886802434921265)]


In [15]:
resultado = model.wv.most_similar(positive=['harry', 'malfoy'], negative=['potter'])
print("Harry - Potter + Malfoy")
print(resultado)

Harry - Potter + Malfoy
[('voz', 0.9899401664733887), ('mão', 0.9893665313720703), ('cabeça', 0.9893143177032471), ('a', 0.9883247017860413), ('com', 0.987755298614502), ('varinha', 0.9874235391616821), ('trêmula', 0.9873210191726685), ('escancarou', 0.9870203733444214), ('sra.', 0.9869512915611267), ('bateu', 0.986720085144043)]


# Modelo 2

In [16]:
# Aumenta Epochs
model2 = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=0, epochs=30, workers=3)

#### Visualizar o modelo

In [17]:
model2.wv.save_word2vec_format('model_harry2.txt',binary=False)
#projector.tensorflow.org

In [18]:
!python -m gensim.scripts.word2vec2tensor -i model_harry2.txt -o model_harry2

2026-04-29 22:36:16,227 - word2vec2tensor - INFO - running C:\Users\helen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gensim\scripts\word2vec2tensor.py -i model_harry2.txt -o model_harry2
2026-04-29 22:36:16,227 - keyedvectors - INFO - loading projection weights from model_harry2.txt
2026-04-29 22:36:17,371 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (6989, 100) matrix of type float32 from model_harry2.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-04-29T22:36:17.329381', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'load_word2vec_format'}
2026-04-29 22:36:17,868 - word2vec2tensor - INFO - 2D tensor file saved to model_harry2_tensor.tsv
2026-04-29 22:36:17,868 - word2vec2tensor - INFO - Tensor metadata file saved to model_harry2_metadata.tsv
2026-04-29 22:36:1

In [19]:
vetor = model2.wv['harry']
print(vetor)

[-0.47875983 -0.50848204 -0.87553513 -0.7288141  -0.89024335 -0.03383461
  0.48466688  0.09164117 -0.25640675 -0.46058342 -0.64070636 -0.38315618
 -1.0599011  -0.6352445  -0.8355795  -1.8691416   0.5105421   0.3925907
 -0.39141083 -1.2256991   0.5005885  -1.0685581   0.554621    0.20339292
 -0.46513313  0.6504718   0.1290907  -0.03674518 -1.1177874   0.39614078
  1.1022865  -0.6053515   0.25420192 -1.1058282  -1.979746    1.7686665
  0.90964013  1.2683978   0.3703611   1.5757471   1.5677067  -1.0527064
  0.34104535 -0.6441209   0.20488334  0.12161296 -0.59145755  0.6009929
  0.57366776 -0.01425949  1.6361399   0.12833233  0.37394035  1.8620915
 -1.9472265   0.10249779  2.342626   -0.71216935 -0.30019376 -0.7451037
  0.8386598  -0.17703539  0.974629   -1.5994065   0.45119599 -0.4291673
 -0.2393955   0.26644567 -0.92201686 -0.57189065  0.21749783 -0.67183757
  0.54237276  2.3506467   0.20214714 -0.14060529 -0.14384219  1.2018483
 -0.98009664  0.32408664 -0.44594392 -0.35241458  0.8418445

In [20]:
'varinha' in model2.wv

True

### Similariedade

In [21]:
model2.wv.most_similar('harry')

[('ele', 0.5399700403213501),
 ('mione', 0.5062275528907776),
 ('portão', 0.4606599509716034),
 ('colin', 0.44873130321502686),
 ('dobby', 0.4481923282146454),
 ('quirrell', 0.43361034989356995),
 ('neville', 0.43315330147743225),
 ('desesperado', 0.4249172806739807),
 ('riddle', 0.4242292046546936),
 ('gina', 0.4080961048603058)]

In [22]:
model2.wv.most_similar('voldemort')

[('lorde', 0.7137608528137207),
 ('descoberto', 0.6438577771186829),
 ('slytherin', 0.6248467564582825),
 ('saber', 0.6196109056472778),
 ('imaginar', 0.615200400352478),
 ('flamel', 0.6113482713699341),
 ('senhor', 0.6074987053871155),
 ('compreender', 0.6054805517196655),
 ('ter-lhe', 0.6048387289047241),
 ('acontecer', 0.5969299077987671)]

In [23]:
print(model2.wv.similarity('harry', 'snape'))
print(model2.wv.similarity('harry', 'hagrid'))
print(model2.wv.similarity('rony', 'hermione'))
print(model2.wv.similarity('harry', 'draco'))

0.3081443
0.27029964
0.33833268
0.40288603


In [24]:
pairs = [
    ('harry', 'rony'),          
    ('harry', 'hermione'),     
    ('harry', 'voldemort'),    
    ('dumbledore', 'snape'),   
    ('varinha', 'vassoura'),   
    ('grifinória', 'sonserina')
]

for w1, w2 in pairs:
    print('%s \t %s \t% .2f' % (w1, w2, model2.wv.similarity(w1,w2)))

harry 	 rony 	 0.38
harry 	 hermione 	 0.32
harry 	 voldemort 	 0.02
dumbledore 	 snape 	 0.56
varinha 	 vassoura 	 0.61
grifinória 	 sonserina 	 0.79


#### Intruso

In [25]:
model2.wv.doesnt_match(['harry', 'rony', 'hermione', 'draco']) 

'hermione'

In [26]:
model2.wv.doesnt_match(['grifinória', 'sonserina', 'corvinal', 'lufa-lufa', 'harry'])

'harry'

In [27]:
resultado = model2.wv.most_similar(positive=['sonserina', 'harry'], negative=['grifinória'])
print("Harry - Grifinória + Sonserina =")
print(resultado)

Harry - Grifinória + Sonserina =
[('quirrell', 0.5436824560165405), ('ele', 0.4940805435180664), ('trasgo', 0.4459637403488159), ('guardachuva', 0.42456144094467163), ('draco', 0.4224725365638733), ('neville', 0.4220101237297058), ('derreter', 0.40209999680519104), ('dobby', 0.3954584002494812), ('girando', 0.39425429701805115), ('firenze', 0.389007031917572)]


In [28]:
resultado = model2.wv.most_similar(positive=['harry', 'malfoy'], negative=['potter'])
print("Harry - Potter + Malfoy")
print(resultado)

Harry - Potter + Malfoy
[('neville', 0.5474326610565186), ('escapar', 0.49116334319114685), ('gritinho', 0.4844231903553009), ('colin', 0.48201385140419006), ('draco', 0.4732888340950012), ('derreter', 0.45608338713645935), ('repente', 0.4507581889629364), ('vermelho', 0.44912293553352356), ('ele', 0.4435923397541046), ('encaixar', 0.43867582082748413)]


# Modelo 3

In [29]:
# Aumenta Vetor e Janela
model3 = Word2Vec(sentences, vector_size=200, window=15, min_count=2, sg=0, epochs=30, workers=3)

In [30]:
model3.wv.save_word2vec_format('model_harry3.txt',binary=False)
#projector.tensorflow.org

In [31]:
!python -m gensim.scripts.word2vec2tensor -i model_harry3.txt -o model_harry3

2026-04-29 22:36:27,741 - word2vec2tensor - INFO - running C:\Users\helen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gensim\scripts\word2vec2tensor.py -i model_harry3.txt -o model_harry3
2026-04-29 22:36:27,742 - keyedvectors - INFO - loading projection weights from model_harry3.txt
2026-04-29 22:36:29,594 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (6989, 200) matrix of type float32 from model_harry3.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-04-29T22:36:29.532759', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'load_word2vec_format'}
2026-04-29 22:36:30,925 - word2vec2tensor - INFO - 2D tensor file saved to model_harry3_tensor.tsv
2026-04-29 22:36:30,925 - word2vec2tensor - INFO - Tensor metadata file saved to model_harry3_metadata.tsv
2026-04-29 22:36:3

In [32]:
vetor = model3.wv['harry']
print(vetor)

[ 1.1666317   1.3484641   0.54011625  0.38086095 -1.8294736  -1.4343079
  0.16297312  1.1618893  -0.93263406  0.7215917   0.47223708 -1.2150441
 -0.01285071  0.8069944   0.23961316 -1.1191952   0.80413437 -0.27786455
 -0.15972824 -1.8179467   0.68914443 -0.75179935  0.46890065 -0.09202535
 -0.41158545 -0.6352671  -0.77567273 -0.2927779  -0.05776504  1.3769964
  0.5591686   0.8172719   0.36127475  0.5521083   1.3400645  -0.5625862
  1.0281395  -0.18745236  0.8831803  -1.5394578  -0.34335405 -0.74967057
  0.6684781   0.11139666 -0.5766247   1.1699593  -0.8943599  -0.10761433
 -0.6408311   0.43853816 -0.02198404  0.3694178  -0.01644008  0.6702029
  1.4086221  -0.12890917  0.34223703 -0.864566   -0.12339788 -0.49640226
 -0.23667563 -0.25684828 -0.56319624  0.94179904 -0.42307696  0.65083504
 -0.5370876  -0.80790555  0.07350405 -1.596853    1.9493495   0.23645441
  0.0028941   0.02149941  0.21113133  0.33194637  1.0788366  -0.14918657
  0.9126372  -0.88314366  0.63675284  0.8579087  -0.5590

In [33]:
'varinha' in model3.wv

True

### Similariedade

In [34]:
model3.wv.most_similar('harry')

[('ele', 0.4616815149784088),
 ('escarlate', 0.43767520785331726),
 ('preso', 0.41447240114212036),
 ('escapar', 0.4105553925037384),
 ('olhado', 0.4089057147502899),
 ('dobby', 0.4063689708709717),
 ('agarrado', 0.3997659385204315),
 ('pottermore.com', 0.3838292360305786),
 ('repente', 0.37975701689720154),
 ('quirrell', 0.3769199252128601)]

In [35]:
model3.wv.most_similar('voldemort')

[('lorde', 0.7942355275154114),
 ('descoberto', 0.6974884867668152),
 ('destruídos', 0.6816002726554871),
 ('poderes', 0.6745349764823914),
 ('slytherin', 0.6697835922241211),
 ('coisas', 0.6620492339134216),
 ('descendente', 0.6558471322059631),
 ('salazar', 0.6359591484069824),
 ('atacante', 0.6346219182014465),
 ('provavelmente', 0.633400559425354)]

In [36]:
print(model3.wv.similarity('harry', 'snape'))
print(model3.wv.similarity('harry', 'hagrid'))
print(model3.wv.similarity('rony', 'hermione'))
print(model3.wv.similarity('harry', 'draco'))

0.23819569
0.084968396
0.33933955
0.2896524


In [37]:
pairs = [
    ('harry', 'rony'),          
    ('harry', 'hermione'),     
    ('harry', 'voldemort'),    
    ('dumbledore', 'snape'),   
    ('varinha', 'vassoura'),   
    ('grifinória', 'sonserina')
]

for w1, w2 in pairs:
    print('%s \t %s \t% .2f' % (w1, w2, model3.wv.similarity(w1,w2)))

harry 	 rony 	 0.15
harry 	 hermione 	 0.22
harry 	 voldemort 	-0.01
dumbledore 	 snape 	 0.45
varinha 	 vassoura 	 0.62
grifinória 	 sonserina 	 0.76


#### Intruso

In [38]:
model3.wv.doesnt_match(['harry', 'rony', 'hermione', 'draco']) 

'harry'

In [39]:
model3.wv.doesnt_match(['grifinória', 'sonserina', 'corvinal', 'lufa-lufa', 'harry'])

'harry'

In [40]:
resultado = model3.wv.most_similar(positive=['sonserina', 'harry'], negative=['grifinória'])
print("Harry - Grifinória + Sonserina =")
print(resultado)

Harry - Grifinória + Sonserina =
[('quirrell', 0.4969514310359955), ('escarlate', 0.47478973865509033), ('ele', 0.452004611492157), ('pottermore.com', 0.4444497525691986), ('agarrado', 0.4413345158100128), ('escapar', 0.43142440915107727), ('dobby', 0.4176310896873474), ('enchendo-se', 0.4044284224510193), ('torcer', 0.3949480652809143), ('presidente', 0.38699808716773987)]


In [41]:
resultado = model3.wv.most_similar(positive=['harry', 'malfoy'], negative=['potter'])
print("Harry - Potter + Malfoy")
print(resultado)

Harry - Potter + Malfoy
[('neville', 0.5447059273719788), ('desdém', 0.4938242733478546), ('simas', 0.47855180501937866), ('escapar', 0.45929035544395447), ('afundara', 0.44041597843170166), ('draco', 0.43926000595092773), ('vomitar', 0.43912485241889954), ('avançou', 0.4176212549209595), ('encaixar', 0.4152830243110657), ('ganido', 0.4148024618625641)]


# Modelo 4

In [42]:
# Skip-Gram e Limpeza de Ruído
model4 = Word2Vec(sentences, vector_size=200, window=15, min_count=5, sg=1, epochs=30, workers=3)

In [43]:
model4.wv.save_word2vec_format('model_harry4.txt',binary=False)
#projector.tensorflow.org

In [44]:
!python -m gensim.scripts.word2vec2tensor -i model_harry4.txt -o model_harry4

2026-04-29 22:36:48,304 - word2vec2tensor - INFO - running C:\Users\helen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\gensim\scripts\word2vec2tensor.py -i model_harry4.txt -o model_harry4
2026-04-29 22:36:48,304 - keyedvectors - INFO - loading projection weights from model_harry4.txt
2026-04-29 22:36:49,081 - utils - INFO - KeyedVectors lifecycle event {'msg': 'loaded (3247, 200) matrix of type float32 from model_harry4.txt', 'binary': False, 'encoding': 'utf8', 'datetime': '2026-04-29T22:36:49.037967', 'gensim': '4.4.0', 'python': '3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]', 'platform': 'Windows-10-10.0.26200-SP0', 'event': 'load_word2vec_format'}
2026-04-29 22:36:49,463 - word2vec2tensor - INFO - 2D tensor file saved to model_harry4_tensor.tsv
2026-04-29 22:36:49,463 - word2vec2tensor - INFO - Tensor metadata file saved to model_harry4_metadata.tsv
2026-04-29 22:36:4

In [45]:
vetor = model4.wv['harry']
print(vetor)

[ 1.93356320e-01 -4.98526692e-02 -9.58494991e-02 -5.11951260e-02
  9.53138247e-02 -2.52575636e-01 -3.15250099e-01  9.83766541e-02
  7.84100965e-02  3.49133760e-02  2.34591048e-02 -1.65781453e-02
  9.97464284e-02  3.32652852e-02  7.67523274e-02 -3.52532454e-02
  3.94261209e-03 -2.10717544e-01  3.10202450e-01 -2.56786913e-01
  2.85384450e-02 -9.77019072e-02 -1.00252643e-01 -8.81974995e-02
 -2.68405706e-01  1.81103665e-02 -4.21962589e-02  9.31686684e-02
  3.48436803e-01  5.68956928e-03  1.88578755e-01  2.78636485e-01
  5.38909473e-02  8.09568465e-02  1.68081462e-01  1.62315706e-03
  1.15607664e-01 -2.90791187e-02  8.67264122e-02 -1.99638501e-01
  3.80130172e-01  7.64972195e-02 -2.76682258e-01 -6.61761034e-03
  2.36344472e-01 -1.15144022e-01 -4.51892316e-02 -1.57832623e-01
 -2.18143106e-01  2.14236695e-02 -1.62372738e-02  9.98964980e-02
 -2.63450831e-01  6.72877254e-03  1.11742057e-01  2.56536696e-02
 -4.14593630e-02 -1.69269770e-01 -1.03700459e-01 -1.59113199e-01
 -1.39016747e-01 -1.35999

In [46]:
'varinha' in model4.wv

True

### Similariedade

In [47]:
model4.wv.most_similar('harry')

[('o', 0.37642210721969604),
 ('rony', 0.37082260847091675),
 ('a', 0.3649933636188507),
 ('assento', 0.364571750164032),
 ('lívido', 0.36143654584884644),
 ('que', 0.3586840033531189),
 ('sente-se', 0.35762470960617065),
 ('cobriu', 0.3554244339466095),
 ('mate-o', 0.35542237758636475),
 ('ordenou', 0.34830614924430847)]

In [48]:
model4.wv.most_similar('voldemort')

[('lorde', 0.6443017721176147),
 ('triz', 0.4937323033809662),
 ('mate-o', 0.47624266147613525),
 ('lobisomem', 0.4586370289325714),
 ('suor', 0.4578028917312622),
 ('matá-lo', 0.4331555962562561),
 ('salazar', 0.43180665373802185),
 ('adormecido', 0.4179629683494568),
 ('pare', 0.41758665442466736),
 ('porquê', 0.4147515296936035)]

In [49]:
print(model4.wv.similarity('harry', 'snape'))
print(model4.wv.similarity('harry', 'hagrid'))
print(model4.wv.similarity('rony', 'hermione'))
print(model4.wv.similarity('harry', 'draco'))

0.11412441
0.17752135
0.3762319
0.15329038


In [50]:
pairs = [
    ('harry', 'rony'),          
    ('harry', 'hermione'),     
    ('harry', 'voldemort'),    
    ('dumbledore', 'snape'),   
    ('varinha', 'vassoura'),   
    ('grifinória', 'sonserina')
]

for w1, w2 in pairs:
    print('%s \t %s \t% .2f' % (w1, w2, model4.wv.similarity(w1,w2)))

harry 	 rony 	 0.37
harry 	 hermione 	 0.20
harry 	 voldemort 	 0.11
dumbledore 	 snape 	 0.13
varinha 	 vassoura 	 0.16
grifinória 	 sonserina 	 0.39


#### Intruso

In [51]:
model4.wv.doesnt_match(['harry', 'rony', 'hermione', 'draco']) 

'draco'

In [52]:
model4.wv.doesnt_match(['grifinória', 'sonserina', 'corvinal', 'lufa-lufa', 'harry'])

'harry'

In [53]:
resultado = model4.wv.most_similar(positive=['sonserina', 'harry'], negative=['grifinória'])
print("Harry - Grifinória + Sonserina =")
print(resultado)

Harry - Grifinória + Sonserina =
[('próprios', 0.31803950667381287), ('o', 0.3122088313102722), ('risadinhas', 0.310227632522583), ('tirá-lo', 0.30313292145729065), ('pesado', 0.30243340134620667), ('cobriu', 0.2992904782295227), ('esfregando', 0.29842668771743774), ('assento', 0.29723837971687317), ('sentou-se', 0.29413387179374695), ('viu-se', 0.2939724326133728)]


In [54]:
resultado = model4.wv.most_similar(positive=['harry', 'malfoy'], negative=['potter'])
print("Harry - Potter + Malfoy")
print(resultado)

Harry - Potter + Malfoy
[('atordoado', 0.33210426568984985), ('rosado', 0.32901740074157715), ('lúcio', 0.3282562494277954), ('poltrona', 0.3175419569015503), ('horrorizado', 0.3114481568336487), ('contava', 0.3090101480484009), ('caçar', 0.3087075352668762), ('cala', 0.30788692831993103), ('incapaz', 0.30699992179870605), ('erguendo', 0.30673927068710327)]
